<a href="https://colab.research.google.com/github/Abhishek-S0111/Pipeline-to-Gather-Audio-Transcript-Datasets-from-Youtube/blob/main/Helper_Notebook_to_download_Audio_from_YT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install yt-dlp -U
# !apt-get update
!apt-get install -y ffmpeg

import pandas as pd
import yt_dlp

In [46]:
# %%capture #Comment out if you dont want any outputs
def YT_Link_Generator(VIDEO_ID :str) -> str:
    """Generator Function to generate YT_Link from VIDEO_ID"""
    return "https://www.youtube.com/watch?v=" + VIDEO_ID

def YT_VIDEO_ID_generator(VIDEO_LINK:str) -> str:
    """Generator function to generate VIDEO_IDS from Link"""
    return VIDEO_LINK[-11:]

def Download_Audio(video_ID):
    import yt_dlp # Import yt_dlp inside the function for multiprocessing

    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': f'/content/KrishiDarshan/{video_ID}/audio.%(ext)s'
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YT_Link_Generator(video_ID)])

In [47]:
# 1. Open an interactive input box to paste your link or ID
user_input = input("Enter YouTube Link or Video ID: ").strip()

# 2. Verify input is not empty, extract the ID, and download
if user_input:
    video_id = YT_VIDEO_ID_generator(user_input)
    print(f"Starting download for Video ID: {video_id}")
    Download_Audio(video_id)
else:
    print("Error: No link or ID was entered.")

Enter YouTube Link or Video ID: 2ELe4jT7G3Y
Starting download for Video ID: 2ELe4jT7G3Y
[youtube] Extracting URL: https://www.youtube.com/watch?v=2ELe4jT7G3Y
[youtube] 2ELe4jT7G3Y: Downloading webpage


[youtube] 2ELe4jT7G3Y: Downloading android vr player API JSON
[info] 2ELe4jT7G3Y: Downloading 1 format(s): 140
[download] Destination: /content/KrishiDarshan/2ELe4jT7G3Y/audio.m4a
[download] 100% of   22.55MiB in 00:00:00 at 37.53MiB/s  
[FixupM4a] Correcting container of "/content/KrishiDarshan/2ELe4jT7G3Y/audio.m4a"
[ExtractAudio] Destination: /content/KrishiDarshan/2ELe4jT7G3Y/audio.wav
Deleting original file /content/KrishiDarshan/2ELe4jT7G3Y/audio.m4a (pass -k to keep)


In [48]:
!pip install -q google-genai

from google import genai
from google.colab import userdata

# Initialize the unified Gemini client
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# Test the connection using gemini-3.5-flash
response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents='Connection successful.'
)
print(response.text)

Connection established! How can I help you today?


In [51]:
import time

# 1. Path to your downloaded audio file
file_path = '/content/KrishiDarshan/2ELe4jT7G3Y/audio.wav'

print("Uploading audio file to Gemini...")
audio_file = client.files.upload(file=file_path)

# Wait briefly for the file API to finish processing the state
print("Waiting for file processing...")
time.sleep(3)

print("\n--- Generating Transcript ---")
# 2. Generate the initial transcription
transcription_response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=[audio_file, "You are an expert, high-precision audio transcriber specializing in Indian agricultural media, specifically regional broadcasts like Krishi Darshan. Your task is to process the provided audio file and generate a highly detailed, chronological, verbatim transcript in hindi. Adhere strictly to the following execution rules: 1. SPEAKER IDENTIFICATION: Differentiate between speakers whenever the voice changes. If names are mentioned, use them (e.g., [Dr. Sharma]). If names are not explicitly mentioned, use logical identifiers like [Host], [Expert], or [Farmer]. 2. TIMESTAMPING RULES: Insert a precise timestamp at the beginning of EVERY speaker turn. If a single speaker talks continuously for more than 45 seconds, insert a new timestamp at the nearest natural sentence boundary. Use the exact format: [HH:MM:SS] (e.g., [00:04:12]). 3. VERBATIM FIDELITY: Transcribe the words exactly as they are spoken. Do not summarize, clean up grammar, or omit technical jargon, crop varieties, chemical names, or regional farming terms. If a technical word, brand name, or number is ambiguous or poorly pronounced, transcribe it to the best of your ability and append a [?] next to it (e.g., [Azotobacter?]). 4. OUTPUT FORMATTING: Present the final output as a clean, line-by-line log. Do not add markdown headers, introductory text, or conversational explanations. Ensure each entry follows this exact template: [TIMESTAMP] [SPEAKER]: Spoken text goes here. Example Output Line: [00:01:15] [Host]: Namaskar kisan bhaiyon, aaj hum baat karenge kharif ki fasal ke baare mein. [00:01:30] [Expert]: Kisan bhaiyon, is samay naye beejon ka chayan karna bahut mahatvapurna hai.."]
)
print(transcription_response.text)

print("\n--- Starting Conversation Mode ---")
# 3. Create an ongoing chat session with the audio context
chat = client.chats.create(model='gemini-3.5-flash')

# FIXED: Changed 'contents=' to 'message=' to align with the new SDK
chat.send_message(message=[audio_file, "Analyze this audio file. I will now ask you questions about its contents."])

print("Chat initialized! You can now ask questions about the audio below. (Type 'exit' to stop)")

# 4. Interactive loop to converse with the audio
while True:
    user_input = input("\nYour Question: ")
    if user_input.lower() == 'exit':
        print("Ending chat session.")
        break

    if user_input.strip() == '':
        continue

    response = chat.send_message(user_input)
    print(f"\nGemini: {response.text}")

Uploading audio file to Gemini...
Waiting for file processing...

--- Generating Transcript ---
[00:00:26] [Host]: नमस्कार। आप सभी किसान भाई बहनों का स्वागत है कार्यक्रम कृषि दर्शन में। किसान भाइयों, अच्छी फसल हो इसके लिए आप खेतों में दिन-रात कड़ी मेहनत करते हैं और आपकी इस मेहनत में आपके सहयोगी हैं मधुमक्खियां। जी हां, मधुमक्खियां करती हैं खेतों में पोलन क्रिया जिससे फसल अच्छी होती है और किसानों को मिलता है ज्यादा लाभ। बहुत से किसान भाई बहन ऐसे हैं जो केवल मधुमक्खी पालन करके अच्छा लाभ ले रहे हैं। मधुमक्खियां फूलों से रस लाती हैं और किसान भाई-बहन प्रोसेसिंग कर शहद का उत्पादन करते हैं। प्रोसेसिंग के दौरान कोई भी चीज बेकार नहीं जाती। शहद का इस्तेमाल हम खाने के लिए करते हैं तो जो अतिरिक्त वैक्स बचता है उसका इस्तेमाल विभिन्न प्रकार की औषधियों और कॉस्मेटिक को बनाने में किया जाता है। आज हम जानकारी लेंगे मधुमक्खी पालन और उससे संबंधित प्रोसेसिंग के बारे में।

[00:01:17] [Narrator]: इस धरती पर हर जीव का अपना महत्व है। उसकी अपनी भूमिका है। इसी वजह से हमारा ईकोसिस्टम ठीक बना रहता है। मधुमक्खियों क

In [55]:
import os
from google.colab import drive

# 1. Mount Google Drive if it isn't already mounted
try:
    drive.mount('/content/drive')
except Exception:
    pass

# 2. Your precise destination base directory path
base_path = "/content/drive/MyDrive/AnnamAI Tasks/Golden Database for Transcript"

# 3. Automatically inherits the video_id variable from the previous download cell
if 'video_id' not in locals() and 'video_id' not in globals():
    raise NameError("Variable 'video_id' not found. Please make sure to run your interactive download cell first!")

# 4. Dynamically append the video ID folder to your base path
target_folder = os.path.join(base_path, video_id)

# 5. Automatically create the directory if it doesn't exist yet
os.makedirs(target_folder, exist_ok=True)

# 6. Define the full text file destination
transcript_path = os.path.join(target_folder, "transcript.txt")

# 7. Write a new file or completely overwrite/update the old file with the latest text
with open(transcript_path, "w", encoding="utf-8") as f:
    f.write(transcription_response.text)

print(f"✨ Success! transcript.txt file added: {transcript_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✨ Success! transcript.txt file added: /content/drive/MyDrive/AnnamAI Tasks/Golden Database for Transcript/2ELe4jT7G3Y/transcript.txt
